# MLC26 Entity Resolution: cloud runner (Kaggle or Colab)

A thin wrapper. All logic lives in `src/`; this notebook only prepares the VM and calls the same command-line entry points we use locally.

**One-time prep**
1. Locally: `bash code/business_entity_resolution/scripts/pack_data.sh` produces `mlc26_data.tar.zst` (about 0.75 GB) and its `.sha256` file.
2. Upload **both** files:
   * **Kaggle:** as a *private* Dataset named `mlc26-data` (Datasets → New Dataset).
   * **Colab:** to `MyDrive/mlc26/`.
3. Add a read-only GitHub fine-grained token (contents: read, this repo only) as a secret named `GH_TOKEN`:
   * **Kaggle:** Add-ons → Secrets. Also turn on Settings → Internet.
   * **Colab:** the 🔑 Secrets panel.

**Rules**
* Read data from the VM's local disk, never from `/content/drive` directly. The Drive mount is slow for large reads.
* Anything that must survive a disconnect goes to `BER_OUTPUT_DIR`:
  * **Kaggle:** `/kaggle/working`, kept when you *Save Version* (runs headless, up to 12h).
  * **Colab:** Drive.

In [ ]:
BRANCH = 'main'  # or a feature branch under test
import os
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
if ON_KAGGLE:
    ARCHIVE, DATA, OUT = '/kaggle/input/mlc26-data/mlc26_data.tar.zst', '/tmp/data', '/kaggle/working/output'
else:
    ARCHIVE, DATA, OUT = '/content/drive/MyDrive/mlc26/mlc26_data.tar.zst', '/content/data', '/content/drive/MyDrive/mlc26/output'
REPO = '/tmp/MLC26' if ON_KAGGLE else '/content/MLC26'
PROJ = f'{REPO}/code/business_entity_resolution'
print('Kaggle' if ON_KAGGLE else 'Colab', '| archive:', ARCHIVE)

In [ ]:
# Hardware: decides which stages this session can run
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'no GPU'
!nproc; free -g | head -2; df -h /tmp | tail -1

In [ ]:
import subprocess
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GH_TOKEN')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    token = userdata.get('GH_TOKEN')
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://{token}@github.com/rishang19dx/AMZN_MLC2026.git', REPO], check=True)
    # keep the token out of .git/config
    subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', 'https://github.com/rishang19dx/AMZN_MLC2026.git'], check=True)
!git -C {REPO} log --oneline -1

In [ ]:
os.environ['BER_DATA_DIR'] = DATA
os.environ['BER_CACHE_DIR'] = '/tmp/cache'
os.environ['BER_OUTPUT_DIR'] = OUT
!bash {PROJ}/scripts/cloud_setup.sh {ARCHIVE}

## Stages
Run the same commands as locally, for example:

`!cd {PROJ} && python src/evaluate.py --split local_val --candidates $BER_OUTPUT_DIR/local_val/candidate_pairs.tsv`

A stage cell is added here as each stage lands in `src/`.